# Batch Scoring — Predictive Maintenance

Scores every row with the champion **registered model** from the MLflow registry.

**Reads:** `gold_ml_features`, `gold_ml_scoring_meta`, MLflow registry  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
import json
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round
)

spark = SparkSession.builder.getOrCreate()

# Load champion registered model + scoring metadata (flavor-aware: predict_proba)
meta = json.loads(spark.read.table('gold_ml_scoring_meta').collect()[0]['meta_json'])
champion_name = meta['champion_model']
champion_flavor = meta['champion_flavor']
feature_cols = meta['feature_cols']
numeric_features = meta['numeric_features']
cat_cols = meta['cat_cols']
category_maps = meta['category_maps']

uri = f'models:/{champion_name}/latest'
model = mlflow.lightgbm.load_model(uri) if champion_flavor == 'lightgbm' else mlflow.sklearn.load_model(uri)
df = spark.read.table('gold_ml_features')
print(f'Loaded registered model: {champion_name}')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
# Prepare features exactly as in training: named float64 pandas columns
pdf = df.toPandas()
for c in numeric_features:
    pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0.0)
for c in cat_cols:
    pdf[f'{c}_idx'] = pdf[c].astype(str).map(category_maps[c]).fillna(-1).astype('float64')

X_all = pdf[feature_cols].astype('float64')
pdf['probability_positive'] = model.predict_proba(X_all)[:, 1]
pdf['prediction'] = model.predict(X_all).astype('int64')
print(f'Scored {len(pdf):,} rows with {champion_name}')

In [ ]:
# Score every row and derive failure probability + risk level
scored = spark.createDataFrame(pdf)


predictions = (
    scored
    .withColumn('failure_probability', spark_round(col('probability_positive'), 4))
    .withColumn('predicted_failure', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('failure_probability') > 0.8, 'critical')
        .when(col('failure_probability') > 0.6, 'high')
        .when(col('failure_probability') > 0.4, 'medium')
        .otherwise('low')
    )
    .withColumn('scored_at', current_timestamp())
    .select(
        'machine_id', 'production_line', 'sensor_date',
        'avg_temp', 'avg_pressure', 'avg_vibration', 'avg_humidity',
        'temp_range', 'pressure_range', 'vibration_range',
        'anomaly_ratio', 'equipment_age_days',
        'had_failure', 'predicted_failure',
        'failure_probability', 'risk_level',
        'scored_at'
    )
)

predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count()} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-machine risk summary
summary = (
    predictions
    .groupBy('machine_id', 'production_line')
    .agg(
        spark_round(avg('failure_probability'), 4).alias('avg_failure_risk'),
        spark_sum('predicted_failure').alias('predicted_failure_days'),
        count('*').alias('total_days'),
        spark_round(avg('avg_temp'), 2).alias('avg_daily_temp'),
        spark_round(avg('avg_pressure'), 2).alias('avg_daily_pressure'),
        spark_round(avg('avg_vibration'), 4).alias('avg_daily_vibration'),
    )
    .withColumn('failure_rate', spark_round(col('predicted_failure_days') / col('total_days') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_failure_risk') > 0.6, 'high')
        .when(col('avg_failure_risk') > 0.3, 'medium')
        .otherwise('low')
    )
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_failure_risk').desc())
)

summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Machine risk summary: {summary.count()} machines')
summary.show(10, truncate=False)

In [ ]:
# Optimize
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('\nAll Gold ML tables optimized')

print('\n=== Scoring Summary ===')
print(f'Total predictions: {predictions.count():,}')
print(f'Machines scored: {summary.count()}')
high_risk = summary.filter(col('overall_risk') == 'high').count()
print(f'High-risk machines: {high_risk}')
print('\nGold tables created:')
print('  - gold_ml_features')
print('  - gold_ml_model_metrics')
print('  - gold_ml_feature_importance')
print('  - gold_ml_predictions')
print('  - gold_ml_summary')